In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project


In [2]:
import subprocess
subprocess.run(["pip", "install", "owlready2"], check=True)

CompletedProcess(args=['pip', 'install', 'owlready2'], returncode=0)

In [4]:
import os

# cherche family.owl dans le projet
for root, dirs, files in os.walk(".."):
    for f in files:
        if "family" in f.lower():
            print(os.path.join(root, f))

..\data\onthologies\family.owl


In [5]:
from owlready2 import get_ontology

onto = get_ontology("../data/onthologies/family.owl").load()

print("Classes:")
for cls in onto.classes():
    print(f"  {cls.name}")

print("\nProperties:")
for prop in onto.properties():
    print(f"  {prop.name}")

print("\nIndividuals:")
for ind in onto.individuals():
    print(f"  {ind.name}")

Classes:
  Son
  Child
  Daughter
  Person
  Uncle
  Parent
  Male
  Grandmother
  Grandparents
  Female
  Grandfather
  Father
  Mother
  Sibling
  Brother
  Sister

Properties:
  age
  nationality
  name
  isSonOf
  isBrotherOf
  isMotherOf
  isFatherOf
  isDaughterOf
  isSisterOf
  isSiblingOf
  isChildOf
  isParentOf
  isMarriedWith

Individuals:


In [9]:
from owlready2 import get_ontology
from owlready2.rule import Imp

onto = get_ontology("../data/onthologies/family.owl").load()

# 1. ajoute la classe oldPerson
with onto:
    class oldPerson(onto.Person):
        pass

# 2. ajoute des individus de test
with onto:
    alice = onto.Person("Alice")
    alice.age = 72
    
    bob = onto.Person("Bob")
    bob.age = 45
    
    charlie = onto.Person("Charlie")
    charlie.age = 65

print("Individus créés:")
for ind in onto.individuals():
    print(f"  {ind.name} | age={ind.age}")

Individus créés:
  Alice | age=72
  Bob | age=45
  Charlie | age=65


In [13]:
from owlready2.rule import Imp, ClassAtom, DataRangeAtom, BuiltinAtom
from owlready2 import *

with onto:
    rule = Imp()
    rule.set_as_rule(
        "Person(?p), age(?p, ?a) -> oldPerson(?p)"
    )

print("Règle SWRL créée ")

Règle SWRL créée 


In [14]:
# règle SWRL simulée : Person(?p) ∧ age(?p, ?a) ∧ ?a > 60 → oldPerson(?p)
print("Règle SWRL : Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → oldPerson(?p)")
print()

with onto:
    for ind in onto.Person.instances():
        if ind.age and ind.age > 60:
            ind.is_a.append(onto.oldPerson)
            print(f"  {ind.name} (age={ind.age}) → classifié comme oldPerson ")
        else:
            print(f"  {ind.name} (age={ind.age}) → pas oldPerson")

print("\noldPersons inférés:")
for ind in onto.oldPerson.instances():
    print(f"  {ind.name} | age={ind.age}")

Règle SWRL : Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → oldPerson(?p)

  Alice (age=72) → classifié comme oldPerson 
  Bob (age=45) → pas oldPerson
  Charlie (age=65) → classifié comme oldPerson 
  Tom (age=10) → pas oldPerson
  Thomas (age=40) → pas oldPerson
  Michael (age=5) → pas oldPerson
  Peter (age=70) → classifié comme oldPerson 
  John (age=45) → pas oldPerson
  Pedro (age=10) → pas oldPerson
  Paul (age=38) → pas oldPerson
  Alex (age=25) → pas oldPerson
  Marie (age=69) → classifié comme oldPerson 
  Sylvie (age=30) → pas oldPerson
  Claude (age=5) → pas oldPerson
  Chloé (age=18) → pas oldPerson

oldPersons inférés:
  Peter | age=70
  Marie | age=69
  Alice | age=72
  Charlie | age=65


In [15]:
onto.save(file="../data/onthologies/family_with_rules.owl", format="rdfxml")
print("Sauvegardé → data/onthologies/family_with_rules.owl")

Sauvegardé → data/onthologies/family_with_rules.owl


In [17]:
from rdflib import Graph

# convertit full_kb.ttl en rdf/xml pour OWLReady2
g = Graph()
g.parse("../kg_artifacts/full_kb.ttl", format="turtle")
g.serialize("../kg_artifacts/full_kb.xml", format="xml")
print(f"Converti → full_kb.xml ({len(g)} triples)")

Converti → full_kb.xml (42219 triples)


In [19]:
from owlready2 import get_ontology, ObjectProperty, Thing
from owlready2.rule import Imp
from rdflib import Graph

# charge le KB F1
f1_onto = get_ontology("../kg_artifacts/full_kb.xml").load()

print(f"Ontologie chargée")
print(f"Classes: {list(f1_onto.classes())[:10]}")
print(f"Propriétés: {list(f1_onto.properties())[:10]}")
print(f"Individus: {len(list(f1_onto.individuals()))}")

Ontologie chargée
Classes: []
Propriétés: []
Individus: 0


In [20]:
from owlready2 import get_ontology, World
from rdflib import Graph

# charge l'ontologie F1 propre
f1_onto = get_ontology("../kg_artifacts/ontology.ttl").load()

print("Classes:", [c.name for c in f1_onto.classes()])
print("Properties:", [p.name for p in f1_onto.properties()])

OwlReadyOntologyParsingError: NTriples parsing error (or unrecognized file format) in ../kg_artifacts/ontology.ttl.

In [21]:
from rdflib import Graph

# convertit ontology.ttl en rdf/xml
g = Graph()
g.parse("../kg_artifacts/ontology.ttl", format="turtle")
g.serialize("../kg_artifacts/ontology.xml", format="xml")
print(f"Converti → ontology.xml ({len(g)} triples)")

Converti → ontology.xml (40 triples)


In [24]:
from owlready2 import get_ontology
from pathlib import Path

path = str(Path("../kg_artifacts/ontology.xml").resolve())
print("Path:", path)

f1_onto = get_ontology(path).load()

print("Classes:", [c.name for c in f1_onto.classes()])
print("Properties:", [p.name for p in f1_onto.properties()])

Path: C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\kg_artifacts\ontology.xml
Classes: ['Driver', 'GrandPrix', 'TeamOrOrganization', 'Season', 'Team', 'CountryOrPlace']
Properties: ['alignmentConfidence', 'isPartOfSeason', 'wonConstructorsChampionshipIn', 'wonGrandPrix']


In [25]:
from owlready2 import get_ontology
from pathlib import Path

with f1_onto:
    # drivers
    lewis = f1_onto.Driver("Lewis_Hamilton")
    max_v = f1_onto.Driver("Max_Verstappen")
    charles = f1_onto.Driver("Charles_Leclerc")
    
    # grand prix
    monaco_2024 = f1_onto.GrandPrix("Monaco_Grand_Prix_2024")
    british_2024 = f1_onto.GrandPrix("British_Grand_Prix_2024")
    
    # saisons
    s2024 = f1_onto.Season("2024")
    s2023 = f1_onto.Season("2023")
    
    # relations
    lewis.wonGrandPrix.append(british_2024)
    charles.wonGrandPrix.append(monaco_2024)
    monaco_2024.isPartOfSeason.append(s2024)
    british_2024.isPartOfSeason.append(s2024)

print("Individus créés ")
print(f"Lewis wonGrandPrix: {lewis.wonGrandPrix}")
print(f"Monaco isPartOfSeason: {monaco_2024.isPartOfSeason}")

Individus créés 
Lewis wonGrandPrix: [C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\kg_artifacts\ontology.xml.British_Grand_Prix_2024]
Monaco isPartOfSeason: [C:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project\kg_artifacts\ontology.xml.2024]


In [26]:
from owlready2.rule import Imp

with f1_onto:
    # crée la nouvelle propriété
    class competedInSeason(f1_onto.Driver >> f1_onto.Season):
        pass

    # règle SWRL
    rule = Imp()
    rule.set_as_rule(
        "Driver(?d), wonGrandPrix(?d, ?gp), isPartOfSeason(?gp, ?s) -> competedInSeason(?d, ?s)",
        namespaces=[f1_onto]
    )

print("Règle SWRL F1 créée ")
print("Driver(?d) ∧ wonGrandPrix(?d, ?gp) ∧ isPartOfSeason(?gp, ?s) → competedInSeason(?d, ?s)")

# applique manuellement la règle
print("\nApplication de la règle:")
with f1_onto:
    for driver in f1_onto.Driver.instances():
        for gp in driver.wonGrandPrix:
            for season in gp.isPartOfSeason:
                if season not in driver.competedInSeason:
                    driver.competedInSeason.append(season)
                    print(f"  {driver.name} → competedInSeason → {season.name} ✅")

print("\nRésultats finaux:")
for driver in f1_onto.Driver.instances():
    if driver.competedInSeason:
        print(f"  {driver.name} : {[s.name for s in driver.competedInSeason]}")

Règle SWRL F1 créée 
Driver(?d) ∧ wonGrandPrix(?d, ?gp) ∧ isPartOfSeason(?gp, ?s) → competedInSeason(?d, ?s)

Application de la règle:
  Oscar_Piastri → competedInSeason → 2017 ✅
  Oscar_Piastri → competedInSeason → 2019 ✅
  Oscar_Piastri → competedInSeason → 2022 ✅
  Oscar_Piastri → competedInSeason → 2023 ✅
  Oscar_Piastri → competedInSeason → 2024 ✅
  Juan_Manuel_Fangio → competedInSeason → 1950 ✅
  Juan_Manuel_Fangio → competedInSeason → 1954 ✅
  Juan_Manuel_Fangio → competedInSeason → 1955 ✅
  Juan_Manuel_Fangio → competedInSeason → 1961 ✅
  Juan_Manuel_Fangio → competedInSeason → 2010 ✅
  Juan_Manuel_Fangio → competedInSeason → 2018 ✅
  Juan_Manuel_Fangio → competedInSeason → 2021 ✅
  Juan_Manuel_Fangio → competedInSeason → 1957 ✅
  Juan_Manuel_Fangio → competedInSeason → 2026 ✅
  Alberto_Ascari → competedInSeason → 1931 ✅
  Alberto_Ascari → competedInSeason → 1952 ✅
  Alberto_Ascari → competedInSeason → 1968 ✅
  Alberto_Ascari → competedInSeason → 1988 ✅
  Alberto_Ascari → compe

In [27]:
f1_onto.save(
    file=str(Path("../kg_artifacts/f1_ontology_with_rules.xml").resolve()),
    format="rdfxml"
)
print("Sauvegardé → kg_artifacts/f1_ontology_with_rules.xml ✅")

Sauvegardé → kg_artifacts/f1_ontology_with_rules.xml ✅


In [29]:
from src.reason.pipeline import run_reasoning_pipeline

stats = run_reasoning_pipeline(
    ontology_xml_path="../kg_artifacts/ontology.xml",
    kb_ttl_path="../kg_artifacts/full_kb.ttl",
    output_path="../kg_artifacts/f1_ontology_with_rules.xml",
)

print(f"\nTotal inférences : {stats['total_inferences']}")
print(f"Drivers inférés  : {stats['drivers_inferred']}")
print("\nExemples:")
for r in stats["inferences"][:5]:
    print(f"  {r['driver']} → {r['season']} via {r['grand_prix']}")

[reasoning] Chargement de l'ontologie...
[reasoning] Chargement des individus depuis le KB...
[reasoning] 932 individus chargés
[reasoning] Ajout de la règle SWRL...
[reasoning] Application des règles...
[reasoning] 146 nouvelles inférences
[reasoning] Sauvegardé → ../kg_artifacts/f1_ontology_with_rules.xml

Total inférences : 146
Drivers inférés  : 17

Exemples:
  Lewis_Hamilton → 2017 via Malaysian_Grand_Prix
  Lewis_Hamilton → 2018 via Japanese_Grand_Prix
  Lewis_Hamilton → 2022 via Japanese_Grand_Prix
  Lewis_Hamilton → 2023 via Japanese_Grand_Prix
  Lewis_Hamilton → 2016 via Abu_Dhabi_Grand_Prix


In [30]:
import os
print("src/reason/:")
for f in sorted(os.listdir("../src/reason")):
    print(f"  {f}")

print("\nkg_artifacts/:")
for f in sorted(os.listdir("../kg_artifacts")):
    print(f"  {f}")

src/reason/:
  __init__.py
  __pycache__
  pipeline.py

kg_artifacts/:
  alignment.ttl
  entity_alignment_table.csv
  expanded.nt
  f1_ontology_with_rules.xml
  full_kb.ttl
  full_kb.xml
  initial_graph.ttl
  kb_stats.txt
  ontology.ttl
  ontology.xml
